<a href="https://colab.research.google.com/github/RicheTasnim/DeepLearningProjects/blob/main/HealtcareChatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1: Import Required Libraries

import json
import random
import pickle
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

In [3]:
#2. Load the Dataset

with open("/content/healthcare_intents.json") as f:
  data = json.load(f)

patterns = []
labels = []

for intent in data["intents"]:
  for pattern in intent["patterns"]:
    patterns.append(pattern.lower())
    labels.append(intent["tag"])

In [4]:
# 3. Encode Intent Labels

encoder = LabelEncoder()
y = encoder.fit_transform(labels)

In [5]:
# 4. Tokenize Text Data

tokenizer = Tokenizer(
    num_words= 5000,
    oov_token='<OOV>'
)

tokenizer.fit_on_texts(patterns)

X = tokenizer.texts_to_sequences(patterns)
X = pad_sequences(
    X,
    maxlen=20,
    padding="post"
)

In [6]:
# 5. Build the LSTM Model

model = Sequential([
    Embedding(
        input_dim=5000,
        output_dim=64,
        input_length=20
    ),

    LSTM(128),

    Dropout(0.5),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        len(np.unique(y)),
        activation="softmax"
    )
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [7]:
# 6. Compile the Model

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [8]:
# 7. Train the Model

history = model.fit(
    X,
    y,
    epochs=100,
    batch_size=16,
    validation_split=0.2
)

Epoch 1/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step - accuracy: 0.0312 - loss: 3.4968 - val_accuracy: 0.0000e+00 - val_loss: 4.1097
Epoch 2/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.0368 - loss: 3.4345 - val_accuracy: 0.0000e+00 - val_loss: 4.7233
Epoch 3/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.0349 - loss: 3.3843 - val_accuracy: 0.0000e+00 - val_loss: 4.6995
Epoch 4/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.0423 - loss: 3.3537 - val_accuracy: 0.0000e+00 - val_loss: 5.4927
Epoch 5/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.0404 - loss: 3.2744 - val_accuracy: 0.0000e+00 - val_loss: 5.5304
Epoch 6/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.0625 - loss: 3.0412 - val_accuracy: 0.0000e+00 - val_loss: 8.3471
Epoch 7/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.0846 - loss: 2.9148 - val_accuracy: 0.0000e+00 - val_loss: 8.8000
Epoch 8/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.1085 - los

In [9]:
# 8. Save the Trained Model

model.save("healthcare_chatbot_models.h5")

pickle.dump(
    tokenizer,
    open("tokenizer.pkl", "wb")
)

pickle.dump(
    encoder,
    open("label_encoder.pkl", "wb")
)

In [14]:
from typing import Sequence
# 9. Create Intent Prediction Function

def predict_intent(text):

  sequence = tokenizer.texts_to_sequences([text])

  padded = pad_sequences(
      sequence,
      maxlen=20,
      padding="post"
  )
  prediction = model.predict(
      padded,
      verbose=0
  )
  index = np.argmax(prediction)

  return encoder.inverse_transform(
      [index]
  )[0]

In [11]:
# 10. Create Response Function

def get_response(intent):
  for item in data["intents"]:
    if item["tag"] == intent:
      return random.choice(
          item["responses"]
      )
  return "Please, consult a heathcare professional."

In [15]:
# 11. Test the Chatbot

while True:

    user_input = input("You: ")

    if user_input.lower() == "quit":
        break

    intent = predict_intent(user_input)

    response = get_response(intent)

    print("Bot:", response)

You: hello
Bot: Hi there! I'm here to assist with your health-related questions.
You: i have bloated stomach
Bot: Consult a healthcare professional if symptoms persist.
You: i have migraine
Bot: Avoid excessive screen time and get adequate rest.
You: i have fever
Bot: Seek medical attention if your fever becomes very high.
You: quit
